# SOMA Uniform vs Proportional — BVH Forward Kinematics Visualization

Visualises the **same motion** (`jump_and_land_heavy_001__A001`) captured under two SOMA body models:

| Model | Body shape | Path |
|---|---|---|
| **Uniform** | Generic human template | `soma_uniform/bvh/…` |
| **Proportional** | Actor-specific body proportions | `soma_proportional/bvh/…` |

The only difference between the two files is the **bone offsets** in the BVH hierarchy —
the same joint-angle sequence produces different 3-D positions when applied to bodies
with different segment lengths.

**Method:** correct BVH parser → forward kinematics (Rz·Ry·Rx Euler accumulation) 
**Output:** `soma_aligned_comparison_fk.gif` (3-panel: Uniform | Proportional | Overlay)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import glob, os, io, time, warnings
from matplotlib.lines import Line2D
from PIL import Image
warnings.filterwarnings('ignore')

# ── Dataset paths ─────────────────────────────────────────────────────────
DATA_ROOT             = '/home/grease/ego_dataset/work_bearlu/data/bones-studio-seed'
SOMA_UNIFORM_DIR      = os.path.join(DATA_ROOT, 'soma_uniform',      'bvh')
SOMA_PROPORTIONAL_DIR = os.path.join(DATA_ROOT, 'soma_proportional', 'bvh')
MOTION_NAME           = 'jump_and_land_heavy_001__A001'

# ── Locate matching BVH files ─────────────────────────────────────────────
uniform_files      = sorted(glob.glob(os.path.join(SOMA_UNIFORM_DIR,      '**/*.bvh'), recursive=True))
proportional_files = sorted(glob.glob(os.path.join(SOMA_PROPORTIONAL_DIR, '**/*.bvh'), recursive=True))

bvh_uniform      = next(f for f in uniform_files      if MOTION_NAME in f)
bvh_proportional = next(f for f in proportional_files if MOTION_NAME in f)

print(f'Uniform      : {bvh_uniform}')
print(f'Proportional : {bvh_proportional}')

In [ ]:
# ── BVH parser ────────────────────────────────────────────────────────────
def parse_bvh(path):
    """
    Parse a BVH file and return the skeleton hierarchy plus all motion frames.

    Critical correctness rule: every JOINT node (including HeadEnd, ThumbEnd …)
    is pushed onto parent_stack.  Only 'End Site' leaf-marker blocks are consumed
    without touching the stack.  Skipping named joints corrupts the stack and
    gives wrong parents for ~40 % of the skeleton.

    Returns
    -------
    joints  : list of joint names (DFS order)
    offsets : {name: np.array([x,y,z])}  T-pose bone offset from parent
    channels: {name: {'start': int, 'types': [str,…]}}
    parents : {name: parent_name or None}
    frames  : np.ndarray  shape (F, total_channels)
    """
    with open(path) as f:
        lines = f.read().split('\n')

    joints, offsets, channels, parents = [], {}, {}, {}
    stack, ch_idx, i = [], 0, 0

    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('ROOT ') or line.startswith('JOINT '):
            name = line.split()[1]
            joints.append(name)
            parents[name] = stack[-1] if stack else None
            stack.append(name)
        elif line.startswith('OFFSET') and stack:
            p = line.split()
            offsets[stack[-1]] = np.array([float(p[1]), float(p[2]), float(p[3])])
        elif line.startswith('CHANNELS') and stack:
            p = line.split(); n = int(p[1])
            channels[stack[-1]] = {'start': ch_idx, 'types': p[2:2+n]}
            ch_idx += n
        elif line.startswith('End Site'):       # leaf marker — consume without stack change
            i += 1
            while i < len(lines):
                if lines[i].strip() == '}':
                    break
                i += 1
        elif line == '}' and stack:
            stack.pop()
        elif line.strip() == 'MOTION':
            break
        i += 1

    mo = next(k for k, l in enumerate(lines) if l.strip() == 'MOTION')
    nf = int(lines[mo+1].split()[1])
    frame_data = []
    for k in range(mo+3, mo+3+nf):
        vals = lines[k].split()
        if vals:
            frame_data.append([float(v) for v in vals])
    return joints, offsets, channels, parents, np.array(frame_data)


# ── Forward kinematics ────────────────────────────────────────────────────
def _rot(angles, types):
    """Rotation matrix from BVH Euler channels (applied in listed order)."""
    def Rx(a): c,s=np.cos(a),np.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    def Ry(a): c,s=np.cos(a),np.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    def Rz(a): c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])
    R = np.eye(3)
    for ang, t in zip(angles, types):
        a = np.radians(ang)
        if t == 'Xrotation': R = R @ Rx(a)
        elif t == 'Yrotation': R = R @ Ry(a)
        elif t == 'Zrotation': R = R @ Rz(a)
    return R

def fk(joints, offsets, channels, parents, frame_data):
    """
    Compute world-space 3-D position for every joint.
    world_pos[j] = world_pos[parent] + world_rot[parent] @ offset[j]
    world_rot[j] = world_rot[parent] @ Rz(z) @ Ry(y) @ Rx(x)
    """
    wp, wr = {}, {}
    for j in joints:
        if j not in channels:
            wp[j] = wp.get(parents.get(j), np.zeros(3)).copy()
            wr[j] = wr.get(parents.get(j), np.eye(3)).copy()
            continue
        ch = channels[j]; start, types = ch['start'], ch['types']
        pos_v = [None]*3; rot_a, rot_t = [], []
        for k, t in enumerate(types):
            v = frame_data[start+k]
            if 'position' in t.lower(): pos_v['XYZ'.index(t[0].upper())] = v
            else: rot_a.append(v); rot_t.append(t)
        local_rot = _rot(rot_a, rot_t)
        parent = parents[j]
        if parent is None:
            wp[j] = np.array([v if v else 0.0 for v in pos_v])
            wr[j] = local_rot
        else:
            local_pos = np.array([v if v else 0.0 for v in pos_v])
            wp[j] = wp.get(parent, np.zeros(3)) + wr.get(parent, np.eye(3)) @ offsets.get(j, np.zeros(3)) + local_pos
            wr[j] = wr.get(parent, np.eye(3)) @ local_rot
    return wp


# ── Body joints used for visualisation (no fingers / face) ────────────────
VIZ_JOINTS = [
    'Hips',
    'Spine1', 'Spine2', 'Chest', 'Neck1', 'Head',
    'LeftShoulder',  'LeftArm',  'LeftForeArm',  'LeftHand',
    'RightShoulder', 'RightArm', 'RightForeArm', 'RightHand',
    'LeftLeg',  'LeftShin',  'LeftFoot',
    'RightLeg', 'RightShin', 'RightFoot',
]

BONES = [
    # Spine
    ('Hips','Spine1'), ('Spine1','Spine2'), ('Spine2','Chest'),
    ('Chest','Neck1'), ('Neck1','Head'),
    # Arms
    ('Chest','LeftShoulder'),  ('LeftShoulder','LeftArm'),   ('LeftArm','LeftForeArm'),   ('LeftForeArm','LeftHand'),
    ('Chest','RightShoulder'), ('RightShoulder','RightArm'), ('RightArm','RightForeArm'), ('RightForeArm','RightHand'),
    # Legs
    ('Hips','LeftLeg'),  ('LeftLeg','LeftShin'),   ('LeftShin','LeftFoot'),
    ('Hips','RightLeg'), ('RightLeg','RightShin'), ('RightShin','RightFoot'),
]


# ── Load BVH and run FK for every sampled frame ───────────────────────────
print('Parsing BVH files…')
j_u, o_u, c_u, p_u, frames_u = parse_bvh(bvh_uniform)
j_p, o_p, c_p, p_p, frames_p = parse_bvh(bvh_proportional)
print(f'  Uniform      : {len(j_u)} joints, {len(frames_u)} frames')
print(f'  Proportional : {len(j_p)} joints, {len(frames_p)} frames')

FRAME_STEP = 4
sampled = list(range(0, min(len(frames_u), len(frames_p)), FRAME_STEP))

def fk_batch(viz, joints, offsets, channels, parents, all_frames, sampled):
    """Return (N_sampled, N_viz_joints, 3) array of world positions."""
    out = np.zeros((len(sampled), len(viz), 3))
    for fi, f in enumerate(sampled):
        wp = fk(joints, offsets, channels, parents, all_frames[f])
        for ji, name in enumerate(viz):
            out[fi, ji] = wp.get(name, np.zeros(3))
    return out

print(f'Running FK for {len(sampled)} sampled frames…')
pos_u = fk_batch(VIZ_JOINTS, j_u, o_u, c_u, p_u, frames_u, sampled)   # (F, J, 3)
pos_p = fk_batch(VIZ_JOINTS, j_p, o_p, c_p, p_p, frames_p, sampled)   # (F, J, 3)

# Floor-align: shift each skeleton so minimum foot Y = 0
foot_idx = [VIZ_JOINTS.index(n) for n in ('LeftFoot', 'RightFoot')]
pos_u[:, :, 1] -= pos_u[:, foot_idx, 1].min()
pos_p[:, :, 1] -= pos_p[:, foot_idx, 1].min()

print(f'  Uniform      Y-range : [{pos_u[:,:,1].min():.0f}, {pos_u[:,:,1].max():.0f}] mm')
print(f'  Proportional Y-range : [{pos_p[:,:,1].min():.0f}, {pos_p[:,:,1].max():.0f}] mm')
print('FK complete ✓')

In [ ]:
# ── Axis limits (fit both skeletons) ──────────────────────────────────────
combined = np.concatenate([pos_u, pos_p], axis=0)
cx = (combined[:,:,0].max() + combined[:,:,0].min()) / 2
cy = (combined[:,:,1].max() + combined[:,:,1].min()) / 2
cz = (combined[:,:,2].max() + combined[:,:,2].min()) / 2
span = max(combined[:,:,0].max() - combined[:,:,0].min(),
           combined[:,:,1].max() - combined[:,:,1].min(),
           combined[:,:,2].max() - combined[:,:,2].min()) * 0.6
xlim = (cx - span, cx + span)
ylim = (0, cy + span)
zlim = (cz - span, cz + span)


# ── Helpers ───────────────────────────────────────────────────────────────
def draw_skeleton(ax, pos_frame, color, alpha=1.0, lw=2.5):
    """Draw bones + joints.  pos_frame: (N_VIZ_JOINTS, 3) in mm."""
    for b0, b1 in BONES:
        if b0 in VIZ_JOINTS and b1 in VIZ_JOINTS:
            i0, i1 = VIZ_JOINTS.index(b0), VIZ_JOINTS.index(b1)
            p0, p1 = pos_frame[i0], pos_frame[i1]
            ax.plot([p0[0], p1[0]], [p0[2], p1[2]], [p0[1], p1[1]],
                    color=color, lw=lw, alpha=alpha)
    for ji in range(len(VIZ_JOINTS)):
        p = pos_frame[ji]
        ax.scatter([p[0]], [p[2]], [p[1]], color=color, s=18, alpha=alpha, depthshade=False)

def style_ax(ax, title):
    ax.set_xlim(xlim); ax.set_ylim(zlim); ax.set_zlim(ylim)
    ax.set_xlabel('X (mm)', color='w', fontsize=7, labelpad=2)
    ax.set_ylabel('Z (mm)', color='w', fontsize=7, labelpad=2)
    ax.set_zlabel('Height (mm)', color='w', fontsize=7, labelpad=2)
    ax.tick_params(colors='grey', labelsize=5)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False
    ax.grid(True, alpha=0.12)
    ax.set_title(title, color='w', fontsize=11, fontweight='bold', pad=6)
    ax.view_init(elev=12, azim=45)
    ax.set_facecolor('#0d1117')

def render_frame(fi):
    """Render one comparison frame (fi = index into sampled list)."""
    fig = plt.figure(figsize=(15, 5), facecolor='#0d1117')
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    ax2 = fig.add_subplot(1, 3, 2, projection='3d')
    ax3 = fig.add_subplot(1, 3, 3, projection='3d')

    draw_skeleton(ax1, pos_u[fi], '#4fc3f7')
    draw_skeleton(ax2, pos_p[fi], '#ef9a9a')
    draw_skeleton(ax3, pos_u[fi], '#4fc3f7', alpha=0.7, lw=2)
    draw_skeleton(ax3, pos_p[fi], '#ef9a9a', alpha=0.7, lw=2)

    style_ax(ax1, 'SOMA Uniform')
    style_ax(ax2, 'SOMA Proportional')
    style_ax(ax3, 'Overlay')
    ax3.legend(
        handles=[Line2D([0],[0], color='#4fc3f7', lw=2, label='Uniform'),
                 Line2D([0],[0], color='#ef9a9a', lw=2, label='Proportional')],
        loc='upper left', fontsize=8, facecolor='#222233', labelcolor='w', framealpha=0.6)

    frame_num = sampled[fi]
    plt.suptitle(
        f'SOMA Uniform vs Proportional  ·  {MOTION_NAME}\n'
        f'Forward Kinematics (FK)  ·  frame {frame_num}  ·  t={frame_num/120:.2f}s',
        color='w', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(pad=0.4)
    return fig


# ── Generate animated GIF ─────────────────────────────────────────────────
N = len(sampled)
print(f'Generating {N} frames…')
gif_frames = []
t0 = time.time()
for fi in range(N):
    if fi % 60 == 0:
        print(f'  {fi}/{N}  ({time.time()-t0:.0f}s)')
    fig = render_frame(fi)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=90, bbox_inches='tight', facecolor='#0d1117')
    buf.seek(0)
    gif_frames.append(Image.open(buf).copy())
    plt.close(fig)

OUT = '/home/grease/gam/soma_aligned_comparison_fk.gif'
gif_frames[0].save(OUT, save_all=True, append_images=gif_frames[1:],
                   duration=50, loop=0, optimize=False)
mb = os.path.getsize(OUT) / 1e6
print(f'\n✅  {OUT}')
print(f'   {mb:.1f} MB  ·  {N} frames  ·  {time.time()-t0:.0f}s to generate')